In [1]:
# ==========================================
# 1. ИМПОРТЫ И НАСТРОЙКИ
# ==========================================
import os
import pickle
import tempfile
import subprocess
import tkinter as tk
import time
import re

import numpy as np
import faiss
import fitz  # PyMuPDF
import httpx

from tkinter import filedialog

INDEX_PATH = "my_index.faiss"
CHUNKS_PATH = "my_chunks.pkl"

OLLAMA_BASE_URL = "http://localhost:11434"
EMBED_MODEL = "nomic-embed-text"
LLM_MODEL = "llama3.2:3b"

CHUNK_SIZE = 900
CHUNK_OVERLAP = 150

DJVUTXT_CMD = "djvutxt"
# ------------------------------------------
# НАСТРОЙКИ ОЧИСТКИ ТЕКСТА ЧЕРЕЗ LLM
# ------------------------------------------
USE_LLM_CLEANING = False
LLM_CLEAN_CHUNK_SIZE = 6000
LLM_CLEAN_OVERLAP = 300
LLM_OPTIONS = {
    "temperature": 0.2,
    "top_p": 0.9,
    "repeat_penalty": 1.1,
}

In [2]:
# ==========================================
# 2. ЧАНКИНГ
# ==========================================
def chunk_text(text: str, size: int = CHUNK_SIZE, overlap: int = CHUNK_OVERLAP) -> list[str]:
    chunks = []
    i = 0
    while i < len(text):
        chunks.append(text[i:i+size])
        i += size - overlap
    return chunks

In [3]:
# ==========================================
# 3. ИЗВЛЕЧЕНИЕ ТЕКСТА ИЗ PDF
# ==========================================
def extract_text_from_pdf(pdf_bytes: bytes) -> str:
    parts = []
    doc = fitz.open(stream=pdf_bytes, filetype='pdf')

    for page in doc:
        txt = page.get_text('text')
        if txt:
            parts.append(txt)

    doc.close()
    return '\n'.join(parts)

In [4]:
# ==========================================
# 4. ИЗВЛЕЧЕНИЕ ТЕКСТОВОГО СЛОЯ ИЗ DJVU
# ==========================================
def extract_text_from_djvu(djvu_bytes: bytes) -> str:
    temp_path = None
    try:
        with tempfile.NamedTemporaryFile(delete=False, suffix='.djvu') as tmp:
            tmp.write(djvu_bytes)
            temp_path = tmp.name

        result = subprocess.run(
            [DJVUTXT_CMD, temp_path],
            stdout=subprocess.PIPE,
            stderr=subprocess.PIPE,
            check=True
        )

        text = result.stdout.decode('utf-8', errors='ignore').strip()
        return text

    except FileNotFoundError:
        raise RuntimeError(
            "Команда djvutxt не найдена. Установи DjVuLibre или укажи полный путь в DJVUTXT_CMD."
        )
    except subprocess.CalledProcessError as e:
        err = e.stderr.decode('utf-8', errors='ignore')
        raise RuntimeError(f'Ошибка djvutxt: {err}')
    finally:
        if temp_path and os.path.exists(temp_path):
            try:
                os.remove(temp_path)
            except:
                pass

In [5]:
# ==========================================
# 5. ОБЩЕЕ ИЗВЛЕЧЕНИЕ ТЕКСТА
# ========================================== 
def extract_text(filename: str, file_bytes: bytes) -> str:
    lower_name = filename.lower()

    if lower_name.endswith('.pdf'):
        return extract_text_from_pdf(file_bytes)

    if lower_name.endswith('.txt'):
        return file_bytes.decode('utf-8', errors='ignore')

    if lower_name.endswith('.djvu') or lower_name.endswith('.djv'):
        return extract_text_from_djvu(file_bytes)

    raise ValueError("Поддерживаются только PDF, TXT и DJVU")     

In [6]:
# ==========================================
# 6. ОЧИСТКА ТЕКСТА (DJVU / OCR)
# ==========================================
def clean_text(text: str) -> str:

    if not text:
        return ""

    # убрать лишние пробелы
    text = re.sub(r"[ \t]+", " ", text)

    # убрать слишком длинные пустые интервалы
    text = re.sub(r"\n{3,}", "\n\n", text)

    # убрать часть мусорных символов
    text = text.replace("_", "")
    text = text.replace("|", "")
    text = text.replace("¬", "")
    text = text.replace("\xad", "")   # мягкий перенос
    text = text.replace("\ufeff", "") # BOM

    return text.strip()

In [7]:
# ==========================================
# 7. LLM-ОЧИСТКА ТЕКСТА ЧЕРЕЗ OLLAMA
# ==========================================
def split_text_for_llm_cleaning(
    text: str,
    chunk_size: int = LLM_CLEAN_CHUNK_SIZE,
    overlap: int = LLM_CLEAN_OVERLAP
) -> list[str]:

    if not text:
        return []

    parts = []
    i = 0

    while i < len(text):
        parts.append(text[i:i + chunk_size])
        i += chunk_size - overlap

    return parts


def llm_clean_chunk(chunk: str, model: str = LLM_MODEL) -> str:

    if not chunk or not chunk.strip():
        return chunk

    system = """
        Ты редактор научного текста на русском языке.
        
        Твоя задача:
        - исправлять явные OCR-ошибки;
        - исправлять смешение русского, английского и других языков, если это не обязательные научные термины;
        - исправлять битые слова, склейки, мусорные символы, лишние пробелы;
        - удалять служебный мусор, если он явно не относится к смыслу текста;
        - сохранять исходный смысл текста;
        - не добавлять новые факты от себя;
        - не пересказывать текст заново;
        - писать строго на русском языке.
        
        Если встречаются общеупотребимые научные термины на английском, можно оставить их только при необходимости.
        Верни только очищенный текст.
        """.strip()

    user = f"""Очисти и нормализуй следующий текст:

        {chunk}
        """

    payload = {
        "model": model,
        "stream": False,
        "messages": [
            {"role": "system", "content": system},
            {"role": "user", "content": user},
        ],
        "options": {
            "temperature": 0
        }
    }

    try:
        r = httpx.post(
            f"{OLLAMA_BASE_URL}/api/chat",
            json=payload,
            timeout=180
        )
        r.raise_for_status()

        data = r.json()
        cleaned = data["message"]["content"].strip()

        if cleaned:
            return cleaned

        return chunk

    except Exception as e:
        print(f"Ошибка LLM-очистки: {type(e).__name__}: {e}")
        return chunk


def llm_clean_text(
    text: str,
    model: str = LLM_MODEL,
    chunk_size: int = LLM_CLEAN_CHUNK_SIZE,
    overlap: int = LLM_CLEAN_OVERLAP
) -> str:

    if not text or not text.strip():
        return text

    parts = split_text_for_llm_cleaning(
        text=text,
        chunk_size=chunk_size,
        overlap=overlap
    )

    if not parts:
        return text

    cleaned_parts = []

    print(f"LLM-очистка: кусков = {len(parts)}")

    for i, part in enumerate(parts, start=1):
        print(f"Очистка текста: {i}/{len(parts)}")
        cleaned_part = llm_clean_chunk(part, model=model)
        cleaned_parts.append(cleaned_part)

    cleaned_text = "\n\n".join(cleaned_parts)
    
    cleaned_text = clean_text(cleaned_text)

    return cleaned_text

In [8]:
# ==========================================
# 8. EMBEDDING ЧЕРЕЗ OLLAMA
# ==========================================
def get_embedding(text: str) -> np.ndarray:
    url = f"{OLLAMA_BASE_URL}/api/embeddings"
    payload = {
        "model": EMBED_MODEL,
        "prompt": text
    }

    r = httpx.post(url, json=payload, timeout=120)
    r.raise_for_status()

    data = r.json()

    if "embedding" not in data:
        raise ValueError(f"В ответе Ollama нет ключа 'embedding': {data}")

    return np.array(data["embedding"], dtype="float32")

In [9]:
# ==========================================
# 9. СОЗДАТЬ ИЛИ ЗАГРУЗИТЬ БАЗУ
# ==========================================
def load_or_create_db():
    if os.path.exists(INDEX_PATH) and os.path.exists(CHUNKS_PATH):
        index = faiss.read_index(INDEX_PATH)

        with open(CHUNKS_PATH, "rb") as f:
            all_chunks = pickle.load(f)

        print(f"База загружена.")
        print(f"Векторов в индексе: {index.ntotal}")
        print(f"Чанков: {len(all_chunks)}")
    else:
        index = None
        all_chunks = []
        print("База не найдена. Она будет создана после загрузки первой книги.")

    return index, all_chunks


index, all_chunks = load_or_create_db()

База загружена.
Векторов в индексе: 45581
Чанков: 45581


In [10]:
# ==========================================
# 10. СОХРАНЕНИЕ БАЗЫ
# ==========================================
def save_db(index, all_chunks):
    faiss.write_index(index, INDEX_PATH)

    with open(CHUNKS_PATH, "wb") as f:
        pickle.dump(all_chunks, f)

    print("База сохранена на диск.")

In [11]:
# ==========================================
# 11. ДОБАВЛЕНИЕ КНИГИ В БАЗУ
# ==========================================
def add_book_to_db(filename: str, file_bytes: bytes):
    global index, all_chunks

    print(f"\nФайл: {filename}")

    try:
        text = extract_text(filename, file_bytes)
        text = clean_text(text)

    except Exception as e:
        print(f"Ошибка извлечения текста: {type(e).__name__}: {e}")
        return

    if not text.strip():
        print("Текст из файла не извлечён.")
        return

    # --------------------------------------
    # LLM-ОЧИСТКА
    # --------------------------------------
    if USE_LLM_CLEANING:
        try:
            print("Запускаю LLM-очистку текста...")
            text = llm_clean_text(text, model=LLM_MODEL)
        except Exception as e:
            print(f"Ошибка LLM-очистки книги: {type(e).__name__}: {e}")
            print("Продолжаю без LLM-очистки...")

    if not text.strip():
        print("После очистки текст пуст.")
        return

    chunks = chunk_text(text)
    print(f"Получено чанков: {len(chunks)}")

    if not chunks:
        print("После чанкинга список пуст.")
        return

    vectors = []

    for i, chunk in enumerate(chunks, start=1):
        try:
            emb = get_embedding(chunk)
            vectors.append(emb)
        except Exception as e:
            print(f"Ошибка embedding на чанке {i}: {type(e).__name__}: {e}")
            return

        if i % 20 == 0 or i == len(chunks):
            print(f"Эмбеддинги: {i}/{len(chunks)}")

    try:
        vectors = np.array(vectors, dtype="float32")
    except Exception as e:
        print(f"Ошибка преобразования vectors в numpy: {type(e).__name__}: {e}")
        return

    if vectors.ndim != 2:
        print("Ошибка: массив векторов должен быть формы (n, dim)")
        print("Текущая форма:", vectors.shape)
        return

    if index is None:
        dim = vectors.shape[1]
        index = faiss.IndexFlatL2(dim)
        print(f"Создан новый индекс FAISS с dim={dim}")
    else:
        if vectors.shape[1] != index.d:
            print("Ошибка: размерность новых векторов не совпадает с существующей базой.")
            print("Размерность новых векторов:", vectors.shape[1])
            print("Размерность базы:", index.d)
            return

    index.add(vectors)

    for chunk in chunks:
        all_chunks.append({
            "book": filename,
            "text": chunk
        })

    print(f"Добавлено векторов: {len(vectors)}")
    print(f"Всего векторов в базе: {index.ntotal}")
    print(f"Всего чанков в базе: {len(all_chunks)}")

    save_db(index, all_chunks)

In [12]:
# ==========================================
# 12. ЗАГРУЗКА КНИГ
# ==========================================
def add_book_from_dialog():
    root = tk.Tk()
    root.withdraw()

    root.attributes('-topmost', True) # окно выбора книги поверх всех окон
    root.update()

    file_path = filedialog.askopenfilename(
        title="Выбери книгу",
        filetypes=[
            ("Books", "*.pdf *.txt *.djvu *.djv"),
            ("PDF files", "*.pdf"),
            ("Text files", "*.txt"),
            ("DjVu files", "*.djvu *.djv"),
            ("All files", "*.*"),
        ]
    )

    root.destroy()

    if not file_path:
        print("Файл не выбран.")
        return

    with open(file_path, "rb") as f:
        content = f.read()

    filename = os.path.basename(file_path)
    print("Файл выбран:", filename)
    print("Размер:", len(content), "байт")

    add_book_to_db(filename, content)

In [13]:
# start = time.time()
# add_book_from_dialog()
# print('Длительность загрузки: ', round((time.time() - start) / 60, 1), ' мин.')

In [14]:
# ==========================================
# 13. ЗАГРУЗКА КНИГИ ПО ПУТИ
# ==========================================
def add_book_from_path(file_path: str):

    if not os.path.exists(file_path):
        print("Файл не найден:", file_path)
        return

    if not os.path.isfile(file_path):
        print("Это не файл:", file_path)
        return

    try:
        with open(file_path, "rb") as f:
            content = f.read()
    except Exception as e:
        print(f"Ошибка чтения файла: {type(e).__name__}: {e}")
        return

    filename = os.path.basename(file_path)

    print("Файл выбран:", filename)
    print("Полный путь:", file_path)
    print("Размер:", len(content), "байт")

    add_book_to_db(filename, content)

In [15]:
base_path = r'D:\КНИГИ\book_torr'
books = os.listdir(base_path)[32:35]

In [66]:
start_parsing = time.time()
for file in books:
    path = os.path.join(base_path, file)
    add_book_from_path(path)
print('Продолжительность: ', round((time.time() - start_parsing) / 60 / 60, 1), ' час')

Файл выбран: 0119 Handbook of optics vol1 - 1995.pdf
Полный путь: D:\КНИГИ\book_torr\0119 Handbook of optics vol1 - 1995.pdf
Размер: 18306843 байт

Файл: 0119 Handbook of optics vol1 - 1995.pdf
Получено чанков: 5369
Эмбеддинги: 20/5369
Эмбеддинги: 40/5369
Эмбеддинги: 60/5369
Эмбеддинги: 80/5369
Эмбеддинги: 100/5369
Эмбеддинги: 120/5369
Эмбеддинги: 140/5369
Эмбеддинги: 160/5369
Эмбеддинги: 180/5369
Эмбеддинги: 200/5369
Эмбеддинги: 220/5369
Эмбеддинги: 240/5369
Эмбеддинги: 260/5369
Эмбеддинги: 280/5369
Эмбеддинги: 300/5369
Эмбеддинги: 320/5369
Эмбеддинги: 340/5369
Эмбеддинги: 360/5369
Эмбеддинги: 380/5369
Эмбеддинги: 400/5369
Эмбеддинги: 420/5369
Эмбеддинги: 440/5369
Эмбеддинги: 460/5369
Эмбеддинги: 480/5369
Эмбеддинги: 500/5369
Эмбеддинги: 520/5369
Эмбеддинги: 540/5369
Эмбеддинги: 560/5369
Эмбеддинги: 580/5369
Эмбеддинги: 600/5369
Эмбеддинги: 620/5369
Эмбеддинги: 640/5369
Эмбеддинги: 660/5369
Эмбеддинги: 680/5369
Эмбеддинги: 700/5369
Эмбеддинги: 720/5369
Эмбеддинги: 740/5369
Эмбеддинги:

In [17]:
# ==========================================
# 14. ПОИСК ПО БАЗЕ
# ==========================================
def search_in_db(query: str, k: int = 5):
    global index, all_chunks

    if index is None or index.ntotal == 0:
        print("База пуста.")
        return []

    q = get_embedding(query).astype("float32").reshape(1, -1)

    if q.shape[1] != index.d:
        print("Ошибка: размерность embedding запроса не совпадает с базой.")
        print("Размерность запроса:", q.shape[1])
        print("Размерность базы:", index.d)
        return []

    D, I = index.search(q, k)

    results = []

    for dist, idx in zip(D[0], I[0]):

        if idx == -1:
            continue

        chunk_data = all_chunks[idx]

        if isinstance(chunk_data, dict):
            results.append({
                "idx": int(idx),
                "distance": float(dist),
                "book": chunk_data.get("book", "unknown_book"),
                "text": chunk_data.get("text", "")
            })
        else:
            results.append({
                "idx": int(idx),
                "distance": float(dist),
                "book": "unknown_book",
                "text": chunk_data
            })

    return results

In [18]:
# ==========================================
# 15. LANGGRAPH: СОСТОЯНИЕ ДЕБАТОВ
# ==========================================
from typing import TypedDict, List
from langgraph.graph import StateGraph, START, END


class DebateState(TypedDict):
    question: str

    retrieved_context: str
    retrieved_items: List[dict]

    hypothesis: str
    criticism: str
    evidence: str

    final_answer: str

In [19]:
# ==========================================
# 16. OLLAMA CHAT
# ==========================================
def ollama_chat(system: str, user: str, model: str = LLM_MODEL) -> str:
    url = f"{OLLAMA_BASE_URL}/api/chat"

    payload = {
        "model": model,
        "stream": False,
        "messages": [
            {"role": "system", "content": system},
            {"role": "user", "content": user},
        ],
        "options": LLM_OPTIONS
    }

    r = httpx.post(url, json=payload, timeout=180)
    r.raise_for_status()

    data = r.json()

    if "message" not in data or "content" not in data["message"]:
        raise ValueError(f"Некорректный ответ Ollama: {data}")

    return data["message"]["content"].strip()

In [20]:
# ==========================================
# 17. ПОДГОТОВКА КОНТЕКСТА ДЛЯ ДЕБАТОВ
# ==========================================
def build_context_from_results(results: list[dict]) -> str:
    if not results:
        return ""

    parts = []

    for i, item in enumerate(results, start=1):
        book = item.get("book", "unknown_book")
        distance = item.get("distance", 0.0)
        text = item.get("text", "").strip()

        part = (
            f"[ФРАГМЕНТ {i}]\n"
            f"Источник: {book}\n"
            f"Distance: {distance:.4f}\n"
            f"Текст:\n{text}"
        )
        parts.append(part)

    return "\n\n".join(parts)

In [21]:
# ==========================================
# 18. УЗЕЛ: ИЗВЛЕЧЕНИЕ КОНТЕКСТА ИЗ БАЗЫ
# ==========================================
def retrieve_context_node(state: DebateState) -> dict:
    question = state["question"]

    results = search_in_db(question, k=5)

    if not results:
        return {
            "retrieved_items": [],
            "retrieved_context": ""
        }

    context = build_context_from_results(results)

    return {
        "retrieved_items": results,
        "retrieved_context": context
    }

In [22]:
# ==========================================
# 19. УЗЕЛ: ВЫДВИЖЕНИЕ ГИПОТЕЗЫ
# ==========================================
def build_hypothesis_node(state: DebateState) -> dict:
    question = state["question"]
    context = state.get("retrieved_context", "")

    if not context.strip():
        return {
            "hypothesis": (
                "Гипотеза не может быть сформулирована, "
                "потому что в базе не найден релевантный контекст."
            )
        }

    system = """
Ты аналитик RAG-системы.

Твоя задача:
- прочитать вопрос;
- прочитать контекст;
- сформулировать предварительную гипотезу;
- опираться только на контекст;
- не выдумывать факты;
- не использовать знания вне контекста;
- если данных мало, прямо скажи об этом.

Пиши на русском языке.
Выводи только текст гипотезы.
"""

    user = f"""
Вопрос:
{question}

Контекст:
{context}

Сформулируй предварительную гипотезу по вопросу строго на основе контекста.
"""

    hypothesis = ollama_chat(system=system, user=user)

    return {"hypothesis": hypothesis}

In [23]:
# ==========================================
# 20. УЗЕЛ: КРИТИКА ГИПОТЕЗЫ
# ==========================================
def critic_review_node(state: DebateState) -> dict:
    question = state["question"]
    context = state.get("retrieved_context", "")
    hypothesis = state.get("hypothesis", "")

    if not context.strip():
        return {
            "criticism": (
                "Критика невозможна, потому что отсутствует найденный контекст."
            )
        }

    system = """
Ты критик-аналитик.

Твоя задача:
- проверить гипотезу на слабые места;
- указать, что подтверждено хорошо;
- указать, что подтверждено слабо;
- отметить противоречия;
- отметить, каких данных не хватает;
- не выдумывать факты;
- использовать только переданный контекст.

Пиши на русском языке.
Выводи только критический разбор.
"""

    user = f"""
Вопрос:
{question}

Контекст:
{context}

Гипотеза:
{hypothesis}

Проведи критический анализ гипотезы.
"""

    criticism = ollama_chat(system=system, user=user)

    return {"criticism": criticism}

In [24]:
# ==========================================
# 21. УЗЕЛ: ДОКАЗАТЕЛЬСТВА
# ==========================================
def evidence_check_node(state: DebateState) -> dict:
    question = state["question"]
    context = state.get("retrieved_context", "")
    hypothesis = state.get("hypothesis", "")
    criticism = state.get("criticism", "")

    if not context.strip():
        return {
            "evidence": (
                "Доказательства отсутствуют, потому что не найден контекст в базе."
            )
        }

    system = """
Ты аналитик доказательств.

Твоя задача:
- выделить подтверждающие доказательства;
- выделить ослабляющие или опровергающие доказательства;
- ссылаться только на переданные фрагменты;
- не придумывать факты;
- если данных недостаточно, прямо укажи это.

Пиши на русском языке.

Формат:
Подтверждает:
- ...

Ослабляет или опровергает:
- ...
"""

    user = f"""
Вопрос:
{question}

Контекст:
{context}

Гипотеза:
{hypothesis}

Критика:
{criticism}

Выдели подтверждающие и ослабляющие доказательства.
"""

    evidence = ollama_chat(system=system, user=user)

    return {"evidence": evidence}

In [25]:
# ==========================================
# 22. УЗЕЛ: ФИНАЛЬНЫЙ ВЫВОД
# ==========================================
def final_conclusion_node(state: DebateState) -> dict:
    question = state["question"]
    context = state.get("retrieved_context", "")
    hypothesis = state.get("hypothesis", "")
    criticism = state.get("criticism", "")
    evidence = state.get("evidence", "")

    system = """
Ты финальный арбитр аналитических дебатов в RAG-системе.

Твоя задача:
- учесть гипотезу;
- учесть критику;
- учесть доказательства;
- сделать итоговый вывод;
- если уверенности мало, сказать об этом прямо;
- не выдумывать факты;
- использовать только контекст.

Пиши на русском языке.
Не смешивай в одном слове английский и русский. Пиши на русском.

Формат ответа строго такой:

ГИПОТЕЗА
...

КРИТИКА ГИПОТЕЗЫ
...

ДОКАЗАТЕЛЬСТВА
Подтверждает:
- ...

Ослабляет или опровергает:
- ...

ИТОГОВЫЙ ВЫВОД
...

УРОВЕНЬ УВЕРЕННОСТИ
высокий / средний / низкий
"""

    user = f"""
Вопрос:
{question}

Контекст:
{context}

Гипотеза:
{hypothesis}

Критика:
{criticism}

Доказательства:
{evidence}

Сформируй финальный ответ.
"""

    final_answer = ollama_chat(system=system, user=user)

    return {"final_answer": final_answer}

In [26]:
# ==========================================
# 23. СБОРКА LANGGRAPH-ГРАФА
# ==========================================
def build_debate_graph():
    graph = StateGraph(DebateState)

    graph.add_node("retrieve_context", retrieve_context_node)
    graph.add_node("build_hypothesis", build_hypothesis_node)
    graph.add_node("critic_review", critic_review_node)
    graph.add_node("evidence_check", evidence_check_node)
    graph.add_node("final_conclusion", final_conclusion_node)

    graph.add_edge(START, "retrieve_context")
    graph.add_edge("retrieve_context", "build_hypothesis")
    graph.add_edge("build_hypothesis", "critic_review")
    graph.add_edge("critic_review", "evidence_check")
    graph.add_edge("evidence_check", "final_conclusion")
    graph.add_edge("final_conclusion", END)

    app = graph.compile()
    return app


debate_app = build_debate_graph()
print("Debate graph готов.")

Debate graph готов.


In [27]:
# ==========================================
# 24. ЗАПУСК ДЕБАТНОГО РЕЖИМА
# ==========================================
def ask_debate_rag(question: str) -> str:
    state = {
        "question": question,
        "retrieved_context": "",
        "retrieved_items": [],
        "hypothesis": "",
        "criticism": "",
        "evidence": "",
        "final_answer": ""
    }

    result = debate_app.invoke(state)
    return result["final_answer"]

In [68]:
# ==========================================
# 25. ПРИМЕР ИСПОЛЬЗОВАНИЯ
# ==========================================
start_debate = time.time()
query = "Предложи технологию, чтобы сделать невидимым человеческому глазу предмет размером несколько метров с помощью метаматериалов"
answer = ask_debate_rag(query)

print('Время на получение ответа: ', round((time.time() - start_debate) / 60, 1), ' мин.')
print(answer)

Время на получение ответа:  1.9  мин.
ГИПОТЕЗА
Гипотеза: Использование метаматериалов с уникальной оптической характеристикой, подобной тем, что наблюдается в редких металлах, может allow сделать предмет размером несколько метров невидимым человеческому глазу.

КРИТИКА
Критический анализ гипотезы:

**Сильные стороны:**

* Использование метаматериалов с уникальной оптической характеристикой может обеспечить высокую эффективность поглощения или отражения света, что может помочь сделать предмет невидимым.

**Слабые стороны:**

* Редкие металлы имеют специфические свойства, которые могут быть сложно имитировать или replicated в метаматерiale.
* Использование метаматериалов с уникальной оптической характеристикой может быть сложным и дорогостоящим процессом.
* Гипотеза не учитывает потенциальные ограничения или противоречия с другими научными теориями или экспериментальными данными.

**Противоречия:**

* Металлический натрий не обнаруживает никаких признаков спектра поглощения, что может оз

In [29]:
# ============================================================
# 26. ПОКАЗАТЬ КНИГИ, КОТОРЫЕ УЖЕ ДОБАВЛЕНЫ В БАЗУ
# ============================================================
def list_books():

    global all_chunks

    if not all_chunks:
        print("База пустая")
        return

    books = set()

    for chunk in all_chunks:
        
        if isinstance(chunk, dict) and "book" in chunk:
            books.add(chunk["book"])

    books = sorted(books)

    print("\nКниги в базе:\n")

    for b in books:
        print("-", b)

    print("\nВсего книг:", len(books))

In [30]:
list_books()


Книги в базе:

- 0009 R Kingslake - Optical system design - 1983.djvu
- 0025 Цернике Ф., Мидвинтер Дж - Прикладная нелинейная оптика - 1976.djvu
- 0090 Скалли М.О. , Зубайри М.С - Квантовая оптика - 2003.djvu
- 0091 С.А Родионов - Автоматизация проектирования оптических систем - 1982.djvu
- 0092 Г.Г Слюсарев - Расчет оптических систем - 1975.djvu
- 0093 Г Агравал - Нелинейная волоконная оптика - 1996.djvu
- 0094 Osche G.R - Optical detection theory for laser applications - 2002.djvu
- 0095 Goodman J. W. - Introduction to Fourier Optics 2nd ed - 1996.djvu
- 0096 Гудмен Дж - Введение в Фурье- оптику - 1970.djvu
- 0098 D Malacara - Optical Shop testing 2nd ed.djvu
- 0099 Ландсберг Г.С. - Оптика. Учебное пособие. 6-е изд - 2003.djvu
- 0100 Справочник технолога-оптика. Под ред. С.М. Кузнецова и М.А Окатова - 1983.djvu
- 0101 Оптические вычисления, под ред. Р Арратуна - 1993.djvu
- 0103 Нагибина И. М - Интерференция и дифракция света - 1985.djvu
- 0104 Mark Fox - Optical Properties of Solid

In [31]:
# ============================================================
# 27. ПОЛНАЯ ОЧИСТКА БАЗЫ
# ============================================================
def clear_db():

    global all_chunks
    global index

    import os

    all_chunks = []
    index.reset()

    # удаление файлов базы
    for f in ["my_index.faiss", "my_chunks.pkl"]:
        
        if os.path.exists(f):
            os.remove(f)
            print("Удалён:", f)

    print("База полностью очищена")

In [32]:
# ==========================================
# 28. ПРОВЕРКА API KEY
# ==========================================
from dotenv import load_dotenv
load_dotenv()
OPENAI_API_KEY = os.getenv("OPENAI_API_KEY")

if not OPENAI_API_KEY:
    raise ValueError("Переменная OPENAI_API_KEY не найдена")

# МАРКЕР

In [9]:
!pip install marker-pdf -q

In [14]:
!pip install git+https://github.com/VikParuchuri/marker

  Running command git clone --filter=blob:none --quiet https://github.com/VikParuchuri/marker 'C:\Users\user\AppData\Local\Temp\pip-req-build-enl_j9rd'



  Cloning https://github.com/VikParuchuri/marker to c:\users\user\appdata\local\temp\pip-req-build-enl_j9rd
  Resolved https://github.com/VikParuchuri/marker to commit d63e3d943b2cbcfd9c809f141f9cdb21294001d5
  Installing build dependencies: started
  Installing build dependencies: finished with status 'done'
  Getting requirements to build wheel: started
  Getting requirements to build wheel: finished with status 'done'
  Preparing metadata (pyproject.toml): started
  Preparing metadata (pyproject.toml): finished with status 'done'


In [28]:
from marker.converters.pdf import PdfConverter

ValueError: Unable to compare versions for numpy>=1.17: need=1.17 found=None. This is unusual. Consider reinstalling numpy.

In [24]:
import numpy
print(numpy.__version__)
print(numpy.__file__)

2.4.2
C:\Users\user\anaconda3\Lib\site-packages\numpy\__init__.py


In [26]:
import sys
print(sys.executable)

C:\Users\user\anaconda3\python.exe


In [ ]:
result = convert_pdf("standart.pdf")

markdown = result["markdown"]

print(markdown[:1000])

ModuleNotFoundError: No module named 'marker.convert'

In [ ]:
convert("Регламент по авансам_organized.pdf", output_dir="output")